# J4-PM — Transformers & Hugging Face

Dans ce notebook on continue sur **SST-2** (la même tâche qu'en J4-AM) pour mesurer le gain apporté par les embeddings contextuels.

| Partie | Méthode | Attendu |
|--------|---------|----------|
| 1 | Explorer le tokenizer HuggingFace | sous-mots, padding, `attention_mask` |
| 2 | La `pipeline` API | sentiment, NER, QA — inférence clé en main |
| 3 | DistilBERT comme extracteur de features | token `[CLS]` + LogReg sklearn |
| 4 | Fine-tuning DistilBERT | `Trainer` end-to-end |
| 5 | Comparaison | GloVe LogReg → BERT frozen → DistilBERT fine-tuné |

In [1]:
!pip install -q transformers datasets evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.2 MB/s eta 0:00:00


In [2]:
import logging
import warnings
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Silence les logs verbeux de HuggingFace
warnings.filterwarnings("ignore")
for logger_name in ("transformers", "datasets", "huggingface_hub"):
    logging.getLogger(logger_name).setLevel(logging.ERROR)

MODEL_NAME = "distilbert-base-uncased"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device : {DEVICE}")

device : cuda


## Partie 1 — Explorer le tokenizer HuggingFace

Le tokenizer `distilbert-base-uncased` utilise **WordPiece** : les mots rares sont découpés en sous-mots connus, préfixés `##`. On charge le tokenizer avec `AutoTokenizer.from_pretrained()` et on observe le découpage sur quelques mots.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

examples = ["unhappiness", "tokenization", "DistilBERT", "cat"]
for word in examples:
    tokens = tokenizer.tokenize(word)
    print(f"{word!r:20s} → {tokens}")

On tokenise maintenant un batch de phrases entières. `padding=True` aligne toutes les séquences sur la plus longue en ajoutant des tokens `[PAD]`. L'`attention_mask` vaut `1` sur les vrais tokens et `0` sur le padding — c'est ce masque que le modèle utilise pour ignorer le padding.

In [4]:
sentences = [
    "The movie was great!",
    "I absolutely hated this film.",
    "Ok.",
]

batch = tokenizer(
    sentences,
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt",
)

print("input_ids      :", batch["input_ids"].shape)   # [3, L]
print("attention_mask :", batch["attention_mask"].shape)
print()
print("input_ids :\n", batch["input_ids"])
print("\nattention_mask (1=vrai token, 0=padding) :\n", batch["attention_mask"])

input_ids      : torch.Size([3, 8])
attention_mask : torch.Size([3, 8])

input_ids :
 tensor([[ 101, 1996, 3185, 2001, 2307,  999,  102,    0],
        [ 101, 1045, 7078, 6283, 2023, 2143, 1012,  102],
        [ 101, 7929, 1012,  102,    0,    0,    0,    0]])

attention_mask (1=vrai token, 0=padding) :
 tensor([[1, 1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 0, 0, 0, 0]])


## Partie 2 — La `pipeline` API

La pipeline `"sentiment-analysis"` télécharge et met en cache automatiquement un DistilBERT pré-entraîné sur SST-2. Elle gère tokenisation, inférence et décodage en une seule ligne.

In [ ]:
from transformers import pipeline

sentiment = pipeline("sentiment-analysis")

phrases = [
    "This movie was absolutely fantastic!",
    "I completely wasted my time.",
    "It was neither good nor bad.",
    "One of the best films I've seen this year.",
    "Terrible acting, dreadful script.",
]
results = sentiment(phrases)
print(f"{'Sentiment':<10} {'Score':>6}  Phrase")
print("-" * 60)
for phrase, result in zip(phrases, results):
    icon = "✅" if result["label"] == "POSITIVE" else "❌"
    print(f"{icon} {result['label']:8s} {result['score']:>5.0%}  {phrase!r}")

À vous d'essayer sur vos propres phrases. Que donne le modèle sur des formulations ambiguës ou ironiques ?

In [ ]:
phrases = [
    "...",
]
for phrase, result in zip(phrases, sentiment(phrases)):
    icon = "✅" if result["label"] == "POSITIVE" else "❌"
    print(f"{icon} {result['label']:8s} {result['score']:>4.0%}  {phrase!r}")

La pipeline `"ner"` détecte les entités nommées (`PER`, `LOC`, `ORG`...). `aggregation_strategy="simple"` regroupe les tokens `##` d'une même entité en un seul span.

In [ ]:
ner = pipeline("ner", aggregation_strategy="simple")

text = (
    "Hugging Face was founded in New York by Clément Delangue and Julien Chaumond in 2016. "
    "The company partners with Google, Microsoft and Amazon."
)
entities = ner(text)
print(f"Texte : {text}\n")
print(f"{'Type':<5}  {'Score':>5}  Entité")
print("-" * 40)
for e in entities:
    print(f"{e['entity_group']:<5}  {e['score']:>4.0%}  {e['word']!r}")

À vous d'essayer sur votre propre texte. Essayez avec des noms francophones, des noms de produits, ou un extrait d'article.

In [ ]:
text = "..."
for e in ner(text):
    print(f"{e['entity_group']:<5}  {e['score']:>4.0%}  {e['word']!r}")

La pipeline `"question-answering"` localise la réponse dans un `context`. Elle attend `question=...` et `context=...` et retourne le span de texte le plus probable avec son `score`.

In [ ]:
qa = pipeline("question-answering")

context = """
Hugging Face is an AI company that develops the Transformers library,
which provides thousands of pre-trained models for natural language processing,
computer vision, and audio tasks. The company was founded in 2016 and is
headquartered in New York City. Their Hub platform hosts over 900,000 models
and 200,000 datasets, used by researchers and developers worldwide.
The Trainer API simplifies fine-tuning of pre-trained models on custom datasets.
"""

questions = [
    "What does Hugging Face develop?",
    "When was Hugging Face founded?",
    "How many models are hosted on the Hub?",
    "What does the Trainer API do?",
]
for q in questions:
    answer = qa(question=q, context=context)
    print(f"Q : {q}")
    print(f"R : {answer['answer']}  ({answer['score']:.0%})\n")

À vous de définir votre propre contexte — une documentation, un article, une fiche produit — et de lui poser des questions. C'est exactement ce que fait un chatbot RAG.

In [ ]:
context = """
...
"""
questions = [
    "...",
]
for q in questions:
    answer = qa(question=q, context=context)
    print(f"Q : {q}\nR : {answer['answer']}  ({answer['score']:.0%})\n")

La `pipeline` supporte de nombreuses autres tâches. Choisissez-en une et testez-la sur un exemple de votre choix.

In [ ]:
# Quelques tâches disponibles — décommentez et adaptez :

# pipe = pipeline("text-generation", model="gpt2")
# print(pipe("Once upon a time", max_new_tokens=50)[0]["generated_text"])

# pipe = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6")
# print(pipe("Long article text here...")[0]["summary_text"])

# pipe = pipeline("translation", model="Helsinki-NLP/opus-mt-fr-en")
# print(pipe("Bonjour, comment ça va ?")[0]["translation_text"])

# pipe = pipeline("fill-mask")
# print(pipe("Hugging Face is a [MASK] company."))


## Partie 3 — DistilBERT comme extracteur de features

In [8]:
ds = load_dataset("glue", "sst2")

# Sous-ensemble pour ne pas attendre trop longtemps
N_TRAIN = 5000
train_texts  = ds["train"]["sentence"][:N_TRAIN]
train_labels = ds["train"]["label"][:N_TRAIN]
val_texts    = ds["validation"]["sentence"]
val_labels   = ds["validation"]["label"]

print(f"train : {len(train_texts)} exemples")
print(f"val   : {len(val_texts)} exemples")

README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

train : 5000 exemples
val   : 872 exemples


On charge DistilBERT avec `AutoModel.from_pretrained()` et on le passe en mode évaluation (`.eval()`). On implémente `extract_cls` qui extrait le vecteur du token `[CLS]` — position `0` de `outputs.last_hidden_state` — pour chaque phrase. Ce vecteur est le résumé de la phrase produit par le modèle.

In [9]:
bert = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
bert.eval()


def extract_cls(texts, tokenizer, model, batch_size=64):
    """Retourne un tableau (N, d) avec le vecteur [CLS] de chaque phrase."""
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = tokenizer(
            texts[i : i + batch_size],
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt",
        )
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.no_grad():
            outputs = model(**batch)
        cls = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(cls)
    return np.vstack(all_embeddings)


print("Extraction train…")
X_train = extract_cls(train_texts, tokenizer, bert)
print("Extraction val…")
X_val = extract_cls(val_texts, tokenizer, bert)
print(f"X_train : {X_train.shape}, X_val : {X_val.shape}")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Extraction train…
Extraction val…
X_train : (5000, 768), X_val : (872, 768)


On entraîne un `LogisticRegression` sklearn sur les vecteurs `[CLS]` — même démarche que sur les embeddings GloVe en J4-AM.

In [10]:
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, train_labels)
acc_frozen = accuracy_score(val_labels, clf.predict(X_val))
print(f"DistilBERT frozen + LogReg : {acc_frozen:.2%}")

DistilBERT frozen + LogReg : 82.11%


## Partie 4 — Fine-tuning DistilBERT

On prépare le dataset avec `.map(tokenize_fn, batched=True)` pour tokeniser toute la colonne `"sentence"`. Le Trainer attend une colonne `"labels"` — on renomme avec `.rename_column("label", "labels")` — et un format `"torch"` fixé avec `.set_format()`.

In [11]:
raw_train = ds["train"].select(range(N_TRAIN))
raw_val   = ds["validation"]


def tokenize_dataset(examples):
    return tokenizer(examples["sentence"], truncation=True, max_length=128)


tok_train = raw_train.map(tokenize_dataset, batched=True)
tok_val   = raw_val.map(tokenize_dataset, batched=True)

# Le Trainer attend une colonne "labels" (et non "label")
tok_train = tok_train.rename_column("label", "labels")
tok_val   = tok_val.rename_column("label", "labels")
tok_train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
tok_val.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

`AutoModelForSequenceClassification` = `AutoModel` (le transformer) + une tête de classification (`nn.Linear(768, num_labels)`). Cette tête joue exactement le rôle de la régression logistique de la partie précédente — sauf qu'elle est intégrée et entraînée bout en bout avec le reste du modèle.

In [12]:
model_ft = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

`TrainingArguments` regroupe tous les hyperparamètres d'entraînement. `Trainer` prend ensuite en charge la boucle complète : shuffling, padding dynamique, gradient accumulation, checkpoints et évaluation automatique à chaque epoch.

In [13]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
)

trainer = Trainer(
    model=model_ft,
    args=training_args,
    train_dataset=tok_train,
    eval_dataset=tok_val,
    data_collator=DataCollatorWithPadding(tokenizer),
)

In [14]:
trainer.train()

{'eval_loss': '0.2721', 'eval_runtime': '1.33', 'eval_samples_per_second': '655.7', 'eval_steps_per_second': '10.53', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': '0.3206', 'eval_runtime': '1.407', 'eval_samples_per_second': '619.6', 'eval_steps_per_second': '9.948', 'epoch': '2'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': '0.3983', 'eval_runtime': '1.498', 'eval_samples_per_second': '582.3', 'eval_steps_per_second': '9.348', 'epoch': '3'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '76.57', 'train_samples_per_second': '195.9', 'train_steps_per_second': '6.151', 'train_loss': '0.2002', 'epoch': '3'}


TrainOutput(global_step=471, training_loss=0.20021909158953688, metrics={'train_runtime': 76.5713, 'train_samples_per_second': 195.896, 'train_steps_per_second': 6.151, 'train_loss': 0.20021909158953688, 'epoch': 3.0})

In [15]:
preds_output = trainer.predict(tok_val)
y_pred = preds_output.predictions.argmax(axis=1)
acc_ft = accuracy_score(val_labels, y_pred)
print(f"DistilBERT fine-tuné : {acc_ft:.2%}")

DistilBERT fine-tuné : 88.65%


On emballe `model_ft` dans une `pipeline("sentiment-analysis", model=..., tokenizer=..., device=...)`. On renseigne `model_ft.config.id2label` pour avoir des labels lisibles (`NEGATIVE` / `POSITIVE`).

In [16]:
model_ft.config.id2label = {0: "NEGATIVE", 1: "POSITIVE"}

ft_pipe = pipeline(
    "sentiment-analysis",
    model=model_ft,
    tokenizer=tokenizer,
    device=0 if DEVICE == "cuda" else -1,
)

test_phrases = [
    "The acting was superb and the story truly gripping.",
    "Boring, predictable, and way too long.",
    "Not bad for a low-budget film.",
    "A masterpiece — I was moved to tears.",
    "Save your money and skip this one.",
]
results = ft_pipe(test_phrases)
print(f"{'Sentiment':<10} {'Score':>5}  Phrase")
print("-" * 65)
for phrase, result in zip(test_phrases, results):
    icon = "✅" if result["label"] == "POSITIVE" else "❌"
    print(f"{icon} {result['label']:8s} {result['score']:>4.0%}  {phrase!r}")

Sentiment  Score  Phrase
-----------------------------------------------------------------
✅ POSITIVE  99%  'The acting was superb and the story truly gripping.'
❌ NEGATIVE  99%  'Boring, predictable, and way too long.'
❌ NEGATIVE  96%  'Not bad for a low-budget film.'
✅ POSITIVE  98%  'A masterpiece — I was moved to tears.'
❌ NEGATIVE  92%  'Save your money and skip this one.'


In [17]:
# À vous ! Testez vos propres phrases
custom_phrases = [
    "Your sentence here...",
]
for phrase, result in zip(custom_phrases, ft_pipe(custom_phrases)):
    icon = "✅" if result["label"] == "POSITIVE" else "❌"
    print(f"{icon} {result['label']:8s} {result['score']:>4.0%}  {phrase!r}")

❌ NEGATIVE  75%  'Your sentence here...'


## Partie 5 — Comparaison

Récapitulatif des accuracies obtenues sur SST-2 sur l'ensemble de la journée.

In [18]:
# Résultats J4-AM (issus du notebook j4am_embeddings_attention)
ACC_GLOVE_FROZEN  = 0.799   # GloVe frozen   + MLP
ACC_GLOVE_FT      = 0.836   # GloVe fine-tuné + MLP

results = {
    "GloVe frozen  (J4-AM)": ACC_GLOVE_FROZEN,
    "GloVe fine-tuné (J4-AM)": ACC_GLOVE_FT,
    "DistilBERT frozen (J4-PM)": acc_frozen,
    "DistilBERT fine-tuné (J4-PM)": acc_ft,
}

print(f"{'Méthode':<35} {'Accuracy':>9}")
print("-" * 46)
for name, acc in results.items():
    print(f"{name:<35} {acc:>8.2%}")

Méthode                              Accuracy
----------------------------------------------
GloVe frozen  (J4-AM)                 79.90%
GloVe fine-tuné (J4-AM)               83.60%
DistilBERT frozen (J4-PM)             82.11%
DistilBERT fine-tuné (J4-PM)          88.65%


### Lecture des résultats

**DistilBERT frozen ≈ GloVe fine-tuné** — c'est contre-intuitif, mais s'explique :
- Le token `[CLS]` d'un modèle **non fine-tuné** n'est pas optimisé pour la classification de sentiment ; c'est une représentation générique.
- GloVe fine-tuné, lui, a pu adapter ses embeddings directement sur la tâche.
- Avec seulement 5 000 exemples d'entraînement, le LogReg n'exploite pas pleinement la richesse des 768 dimensions de BERT.

**DistilBERT fine-tuné** (+6 points) — dès qu'on laisse tous les paramètres s'adapter, le modèle contextuel prend largement l'avantage : chaque token est représenté en fonction de son contexte, et la tête de classification est entraînée de bout en bout.